# Pestaña 4 · Evaluación en el conjunto de prueba

In [1]:
import sys
sys.path.insert(0, "dashboard")
import pandas as pd
from IPython.display import HTML, Markdown, display
from src import figuras as fg
from src import textos as tx
from src.datos import num, objeto, pct, tabla

_primera = [True]


def figura(f):
    """Inserta la figura interactiva de Plotly (la librería se carga una vez por página)."""
    display(HTML(f.to_html(full_html=False, include_plotlyjs="cdn" if _primera[0] else False, config=fg.CONFIG,
                           default_width="100%")))
    _primera[0] = False


def texto(parrafos):
    display(Markdown("\n\n".join(parrafos)))


def tab(df, dec=3):
    display(df.style.format(precision=dec, decimal=",", thousands=" ", na_rep="—").hide(axis="index"))

La cuarta pestaña evalúa una sola vez en la prueba (2009-2010) el mejor representante de cada modelo, reajustado con todo el entrenamiento, junto con el modelo base del Entregable 2.

![Vista de la pestaña en el dashboard](../imagenes/pestana_4_prueba.png)

Las figuras son interactivas (se puede pasar el cursor sobre ellas) y las interpretaciones usan las mismas cifras que el dashboard y que el cuaderno.

## 4.1 Métricas, curvas y matrices de confusión

In [2]:
tab(tabla('finalistas_prueba').drop(columns=['clave', 'auc_inf', 'auc_sup', 'ap_inf', 'ap_sup']))
figura(fg.curvas_prueba('ROC'))
figura(fg.curvas_prueba('PR'))

Modelo,Balanceo,Optimizador,Accuracy,Precision,Recall,F1,AUC,AUC IC 95 %,AUC-PR,AUC-PR IC 95 %
KNN,class_weight,Grid Search,"0,444","0,345","0,895","0,498","0,688","[0,675; 0,701]","0,500","[0,478; 0,518]"
Naive Bayes,class_weight,Bayesiana (Optuna TPE),"0,524","0,377","0,840","0,521","0,714","[0,702; 0,727]","0,534","[0,515; 0,552]"
Logística L1/L2,SMOTE,Random Search,"0,394","0,326","0,907","0,480","0,681","[0,667; 0,695]","0,530","[0,507; 0,550]"
Árbol de decisión,SMOTE,Bayesiana (Optuna TPE),"0,659","0,417","0,271","0,329","0,567","[0,554; 0,580]","0,374","[0,357; 0,390]"
Random Forest,Sin balanceo,Bayesiana (Optuna TPE),"0,692","0,667","0,001","0,002","0,720","[0,708; 0,732]","0,503","[0,482; 0,521]"
XGBoost,class_weight,Genético (DEAP),"0,593","0,352","0,382","0,367","0,559","[0,546; 0,573]","0,369","[0,349; 0,386]"
SVM,class_weight,Grid Search,"0,411","0,337","0,937","0,495","0,724","[0,711; 0,736]","0,538","[0,516; 0,560]"
"Referencia E2 (logística C = 0,01)",Sin balanceo,—,"0,710","0,750","0,087","0,157","0,710","[0,696; 0,721]","0,548","[0,526; 0,568]"


In [3]:
for m_ in [m for m in fg.MODELOS + ['Referencia E2'] if m in set(tabla('curvas_prueba')['modelo'])]:
    figura(fg.matriz_confusion(m_))

In [4]:
texto(tx.prueba())

**Interpretación.** En prueba, el mayor AUC-PR es el de SVM (0,538 [0,516; 0,560], AUC 0,724), frente a 0,548 de la referencia del Entregable 2. La tasa de suscripción en prueba es del 30,8 %, que es el AUC-PR de un clasificador al azar.

Esto significa que los modelos ordenan bien a los clientes en un periodo que no vieron, pero el umbral por defecto, fijado con la tasa del entrenamiento, deja sin marcar a muchos suscriptores en los modelos sin balanceo.

Por ello, se recomienda usar el orden (AUC y AUC-PR) para decidir a quién llamar primero y fijar el punto de operación con probabilidades recalibradas al régimen vigente.

## 4.2 Calibración y recalibración

In [5]:
for m_ in tabla('calibracion_curvas')['modelo'].unique():
    figura(fg.confiabilidad(m_))
tab(tabla('calibracion').drop(columns=['clave']), dec=4)
texto(tx.calibracion())

Modelo,Balanceo,Variante,Brier,ECE,Prob. media,AUC-PR,"F1 (umbral 0,5)"
KNN,class_weight,Sin recalibrar,"0,2990","0,3232","0,6283","0,4998","0,4981"
KNN,class_weight,Platt,"0,2506","0,1939","0,1144","0,3255","0,0000"
KNN,class_weight,Isotónica,"0,2497","0,1942","0,1143","0,3382","0,0000"
KNN,class_weight,Mensual (intercepto),"0,1764","0,0382","0,2834","0,5770","0,4883"
Naive Bayes,class_weight,Sin recalibrar,"0,4049","0,4109","0,7166","0,5343","0,5208"
Naive Bayes,class_weight,Platt,"0,2470","0,2199","0,0883","0,4644","0,0000"
Naive Bayes,class_weight,Isotónica,"0,2263","0,1885","0,1198","0,4356","0,0000"
Naive Bayes,class_weight,Mensual (intercepto),"0,2664","0,2356","0,3446","0,4777","0,5204"
Logística L1/L2,SMOTE,Sin recalibrar,"0,3050","0,3349","0,6432","0,5297","0,4799"
Logística L1/L2,SMOTE,Platt,"0,2459","0,1876","0,1277","0,3681","0,0807"


**Interpretación.** La mediana del ECE en prueba por variante es 0,055 (Mensual (intercepto)), 0,189 (Sin recalibrar), 0,201 (Isotónica) y 0,204 (Platt).

Esto quiere decir que la descalibración está dominada por el cambio de prevalencia: Platt y la isotónica se ajustan con datos del entrenamiento y no anticipan el régimen nuevo, mientras que la recalibración mensual sigue la tasa del último mes cerrado.

Por tal motivo, se opta por la variante Mensual (intercepto) cuando importe el valor de la probabilidad, por ejemplo para estimar cuántas suscripciones dará una campaña.

## 4.3 Regresión: duración de la llamada

In [6]:
tab(tabla('regresion_prueba').drop(columns=['clave']), dec=4)
figura(fg.serie_regresion())
figura(fg.residuos_regresion())
texto(tx.regresion())

Modelo,Optimizador,RMSE (log),MAE (log),RMSE (s),MAE (s),R²,p White,p BDS (m = 2)
KNN,Genético (DEAP),"0,9342","0,6955","258,8261","155,4867","0,0427","0,0000","0,0000"
Ridge,Random Search,"0,9375","0,6927","256,3659","154,8141","0,0360","0,0000","0,0000"
Lasso,Genético (DEAP),"0,9459","0,7089","266,6585","157,9900","0,0186","0,0000","0,0000"
Árbol de decisión,Grid Search,"0,9166","0,6925","258,0030","155,2728","0,0784","0,0000","0,0000"
Random Forest,Random Search,"0,9120","0,6896","261,7158","154,7025","0,0877","0,0000","0,0000"
XGBoost,Bayesiana (Optuna TPE),"0,9298","0,6973","263,0896","156,0172","0,0518","0,0000","0,0000"
SVR,Genético (DEAP),"0,9639","0,7081","253,9521","159,2287","-0,0190","0,0000","0,0000"
Media del entrenamiento (referencia),—,"0,9563","0,7154","268,1270","159,1151","-0,0030",—,—


**Interpretación.** El mejor modelo de regresión (Random Forest) logra un RMSE de 0,9120 en escala log y un R² de 0,088; la media del entrenamiento, 0,9563 y -0,003. En sus residuos, White da p < 0,001 y Jarque-Bera p < 0,001.

Esto significa que la información previa a la llamada explica apenas el 9 % de la varianza de la duración: los modelos superan a la media de forma significativa, pero no anticipan qué llamadas serán muy cortas o muy largas, y los residuos heredan las colas pesadas y la dependencia de la propia duración.

En consecuencia, se decide reportar la rama de regresión como un resultado de señal débil y no usarla para planificar la carga de los asesores llamada a llamada.